In [1]:
from notebookutils import mssparkutils
from datetime import datetime

try:
    batch_id = mssparkutils.widgets.get("batch_id")
except Exception:
    batch_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
print("batch_id = ", batch_id)

StatementMeta(, 87642042-07de-4d09-964d-aec429226c85, 3, Finished, Available, Finished, False)

batch_id =  20260416_181127


In [11]:
from pyspark.sql import functions as F

raw_review_path ="Files/yelp/raw/yelp_review.json"

df_review_raw = spark.read.json(raw_review_path)
df_review_raw.printSchema()
print("raw count = ", df_review_raw.count())


StatementMeta(, 65460109-d8a1-470c-9526-9d7754200d32, 12, Finished, Available, Finished, False)

root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)

raw count =  6990280


In [15]:
df_review_bronze = (
    df_review_raw
    .withColumn("_ingest_ts",F.current_timestamp())
    .withColumn("_source_file",F.input_file_name())
    .withColumn("_batch_id", F.lit(batch_id))
)


StatementMeta(, 65460109-d8a1-470c-9526-9d7754200d32, 16, Finished, Available, Finished, False)

In [16]:
(df_review_bronze.write
.format("delta")
.mode("overwrite")
.saveAsTable("review_bronze")
)

spark.table("review_bronze").count()

StatementMeta(, 65460109-d8a1-470c-9526-9d7754200d32, 17, Finished, Available, Finished, False)

6990280

In [18]:
df_review_bronze.select("business_id","cool","date","funny","review_id","stars","text","useful","user_id","_ingest_ts","_source_file","_batch_id")

StatementMeta(, 65460109-d8a1-470c-9526-9d7754200d32, 19, Finished, Available, Finished, False)

DataFrame[business_id: string, cool: bigint, date: string, funny: bigint, review_id: string, stars: double, text: string, useful: bigint, user_id: string, _ingest_ts: timestamp, _source_file: string, _batch_id: string]

In [19]:
#验证引入完成
spark.table("review_bronze").printSchema()

StatementMeta(, 65460109-d8a1-470c-9526-9d7754200d32, 20, Finished, Available, Finished, False)

root
 |-- business_id: string (nullable = true)
 |-- cool: long (nullable = true)
 |-- date: string (nullable = true)
 |-- funny: long (nullable = true)
 |-- review_id: string (nullable = true)
 |-- stars: double (nullable = true)
 |-- text: string (nullable = true)
 |-- useful: long (nullable = true)
 |-- user_id: string (nullable = true)
 |-- _ingest_ts: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)



In [ ]:
#Return to 01_0_bronze_run_all
mssparkutils.notebook.exit("SUCCESS")